# 01 — Data Extraction

This notebook runs the three standalone SQL files in `sql/` against the AACT-schema Postgres database (raw `psycopg2`/`pandas.read_sql_query`, no ORM — see `src/db.py`), joins the results in pandas, computes `duration_days`, and writes `data/processed/analysis_table.csv`.

This is a thin, readable wrapper around `src/run_extraction.py` — the same logic that file runs — so the extraction step can be inspected and re-run cell-by-cell rather than only as a script.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.db import run_sql_file

SQL_DIR = '../sql'
pd.set_option('display.max_columns', None)

## Step 1 — run each standalone `.sql` file

Each query lives in its own `.sql` file (not embedded in a Python string) so it can be read, run, and explained on its own. See the comments at the top of each `.sql` file for *why* each filter/join was chosen.

In [2]:
studies = run_sql_file(os.path.join(SQL_DIR, 'extract_studies.sql'))
sponsors = run_sql_file(os.path.join(SQL_DIR, 'extract_sponsors.sql'))
conditions = run_sql_file(os.path.join(SQL_DIR, 'extract_conditions.sql'))

print('studies:', studies.shape)
print('sponsors:', sponsors.shape)
print('conditions:', conditions.shape)
studies.head()

studies: (9000, 7)
sponsors: (9000, 3)
conditions: (9000, 2)


/home/claude/TrialScope/src/db.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,nct_id,study_type,phase,overall_status,enrollment,start_date,completion_date
0,NCT100000000,Interventional,Phase 3,Completed,27,2022-08-10,2025-09-24
1,NCT100000001,Interventional,Phase 3,Completed,10,2012-05-30,2015-01-03
2,NCT100000002,Interventional,Phase 1,Completed,33,2012-12-02,2015-10-05
3,NCT100000003,Interventional,Phase 1,"Active, not recruiting",276,2020-08-28,None
4,NCT100000004,Interventional,Phase 2,Unknown status,33,2017-08-26,None


## Step 2 — join into one analysis table

**Lead-sponsor-only** (documented in `extract_sponsors.sql`): the SQL already filtered to one lead-sponsor row per study, so this merge cannot duplicate rows.

**Left joins, not inner**: the SQL files already did the scope filtering (interventional, date range, valid phase). This join should not silently drop anything further — a study with no lead sponsor still appears, with `sponsor_type` as NaN, so missingness stays visible instead of vanishing.

In [3]:
df = studies.merge(sponsors, on='nct_id', how='left')
df = df.merge(conditions, on='nct_id', how='left')
df['start_date'] = pd.to_datetime(df['start_date'])
df['completion_date'] = pd.to_datetime(df['completion_date'])
df.shape

(9000, 10)

## Step 3 — `duration_days`, with explicit handling of ongoing trials

A trial still `Recruiting` / `Active, not recruiting` / `Unknown status` has **no** `completion_date` yet — that's not bad data, it's a trial that hasn't finished. Those rows are **kept** for every other analysis (phase mix, enrollment, status counts) and excluded **only** from anything keyed on `duration_days`. The cell below reports exactly how many rows that affects, rather than silently dropping them everywhere.

In [4]:
has_both_dates = df['start_date'].notna() & df['completion_date'].notna()
df['duration_days'] = pd.NA
df.loc[has_both_dates, 'duration_days'] = (
    df.loc[has_both_dates, 'completion_date'] - df.loc[has_both_dates, 'start_date']
).dt.days

n_missing = int((~has_both_dates).sum())
print(f'{n_missing} of {len(df)} studies ({n_missing/len(df):.1%}) have no '
      f'completion_date and are excluded from duration_days analyses only.')

2824 of 9000 studies (31.4%) have no completion_date and are excluded from duration_days analyses only.


In [5]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/analysis_table.csv', index=False)
df.describe(include='all').T.head(10)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
nct_id,9000,9000,NCT100000000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_type,9000,1,Interventional,9000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phase,9000,4,Phase 2,3061,NaN,NaN,NaN,NaN,NaN,NaN,NaN
overall_status,9000,6,Completed,4727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
enrollment,9000.0,NaN,NaN,NaN,152.295222,5.0,32.0,71.0,163.0,4095.0,256.745382
start_date,9000,NaN,NaN,NaN,2018-06-27 22:16:48,2011-01-01 00:00:00,2014-08-26 00:00:00,2018-07-15 12:00:00,2022-04-08 06:00:00,2025-12-30 00:00:00,NaN
completion_date,6176,NaN,NaN,NaN,2020-06-05 08:00:18,2011-05-21 00:00:00,2016-10-18 18:00:00,2020-06-24 12:00:00,2024-05-04 06:00:00,2026-12-31 00:00:00,NaN
sponsor_type,9000,3,Industry,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sponsor_name,9000,1119,Industry Sponsor #271,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_condition,9000,20,Psoriasis,485,NaN,NaN,NaN,NaN,NaN,NaN,NaN
